script for scores:
- subdf with the top20 scores
- take mean, weighted mean, multiplication (prod) and add to df
- calculate roc,pr,fr... improve? ja/ nein

TOP20: ['chai_constrained_interface_iptm', 'boltz_template_per_chain_pair_iptm', 'boltz_free_confidence_score', 'boltz_template_iptm', 'esmfold_ptm', 'boltz_free_per_chain_pair_iptm', 'boltz_free_iptm', 'boltz_free_pde', 'boltz_template_confidence_score', 'boltz_free_iplddt', 'boltz_template_iplddt', 'chai_unconstrained_per_chain_iptm', 'boltz_free_plddt', 'boltz_template_ptm', 'chai_unconstrained_ptm', 'cf_ipsae', 'chai_unconstrained_aggregate_score', 'chai_unconstrained_score', 'chai_unconstrained_iptm', 'boltz_free_ptm']

In [2]:
import pandas as pd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

CSV_PATH = Path("/home/bri/pymol/EDA/static/scores2combine.csv")
df = pd.read_csv(CSV_PATH)

OUTPUT_DIR = Path("/home/bri/pymol/EDA/static/top_scores")
OUTPUT_DIR.mkdir(exist_ok=True)

In [3]:
df.head(10)

,source,type,binder,KD[M],EC50[M],sample,chai_constrained_ipsae,chai_constrained_interface_iptm,chai_constrained_pdockq,af3_pdockq,...,chai_unconstrained_per_chain_iptm,cf_ipsae,chai_unconstrained_pdockq,chai_unconstrained_ipsae,chai_unconstrained_per_chain_ptm,cf_iplddt,af3_ipsae,cf_pdockq2,cf_pyros_binder_score,esmfold_iplddt
0,mcmahon,original,1,4.310000e-07,NaN,Nb.b201_HSA_404-609,0.828829,0.672600,0.3520,0.2994,...,0.171786,0.717862,0.1053,0.005310,0.886037,95.894167,0.669827,0.8469,8540.50,NaN
1,mcmahon,original,1,4.310000e-07,NaN,Nb.b201_HSA_404-601,0.531820,0.603329,0.1551,0.2819,...,0.147295,0.713582,0.1133,0.005263,0.878948,95.717916,0.625092,0.8463,65086.93,NaN
2,mcmahon,original,1,4.310000e-07,NaN,Nb.b201_HSA_25-609,0.644171,0.670695,0.2906,0.1625,...,0.255400,0.538915,0.2691,0.005231,0.877172,86.205385,0.000000,0.4583,41598.65,63.759457
3,mcmahon,original,1,2.300000e-06,NaN,Nb.AQ100_Adiponectin,0.163092,0.281817,0.0822,0.1487,...,0.159133,0.370364,0.0486,0.028691,0.882637,84.943667,0.006573,0.3299,26952.43,56.575943
4,mcmahon,original,0,NaN,NaN,Nb.AQ101_Adiponectin,0.201847,0.338975,0.2525,0.0573,...,0.171685,0.244249,0.0466,0.040458,0.881152,81.203705,0.005250,0.2045,15987.81,46.311729
5,mcmahon,original,0,NaN,NaN,Nb.AQ102_Adiponectin,0.475568,0.590482,0.1981,0.1164,...,0.238288,0.123757,0.0731,0.110449,0.895664,76.639667,0.005555,0.0286,16335.52,52.373833
6,mcmahon,original,1,5.100000e-07,NaN,Nb.AQ103_Adiponectin,0.235164,0.354226,0.0917,0.0752,...,0.218349,0.421352,0.0587,0.091507,0.896221,84.733914,0.005775,0.3342,18606.78,NaN
7,mcmahon,original,0,NaN,NaN,Nb.AQ104_Adiponectin,0.142033,0.269744,0.1465,0.0706,...,0.141926,0.145388,0.0685,0.005675,0.872838,75.301852,0.006316,0.0475,307.39,50.281277
8,mcmahon,original,0,NaN,NaN,Nb.AD101_A2AR,0.201931,0.321660,0.0807,0.1428,...,0.242299,0.553068,0.1598,0.105844,0.863009,82.785477,0.091485,0.4043,11364.54,49.020782
9,mcmahon,original,0,NaN,NaN,Nb.AD102_A2AR,0.333846,0.440906,0.1960,0.1183,...,0.157157,0.700968,0.2165,0.000000,0.876953,90.295555,0.005404,0.6723,4979.12,49.552746


In [4]:
# normalize scores to 0-1 range (higher = better) --> actually not that good
# use standard scaler

from sklearn.preprocessing import StandardScaler
cols_to_combine = ['chai_constrained_ipsae', 'boltz_template_pdockq'] # they are both between 0-1 --> not necessary, but we can do standard scaling to center them around 0 and have unit variance

# standard scaler
scaler = StandardScaler()
df_norm = pd.DataFrame(scaler.fit_transform(df[cols_to_combine]), columns=cols_to_combine)

df_norm.head()

# NOTE — normalization for ML (future):
#   Scale does not matter for ROC-AUC but will matter for ML models.
#   At that point use QuantileTransformer fit on training data only:
#       qt.fit(X_train[score_cols])
#       X_train_norm = qt.transform(X_train[score_cols])
#       X_test_norm  = qt.transform(X_test[score_cols])  # NOT fit_transform!

,chai_constrained_ipsae,boltz_template_pdockq
0,2.117209,0.803183
1,0.740164,0.369367
2,1.261065,2.456056
3,-0.969397,0.241045
4,-0.789714,-0.672519


In [5]:
# Mean
df['Mean_Score'] = df_norm.mean(axis=1)

df.head()

,source,type,binder,KD[M],EC50[M],sample,chai_constrained_ipsae,chai_constrained_interface_iptm,chai_constrained_pdockq,af3_pdockq,...,cf_ipsae,chai_unconstrained_pdockq,chai_unconstrained_ipsae,chai_unconstrained_per_chain_ptm,cf_iplddt,af3_ipsae,cf_pdockq2,cf_pyros_binder_score,esmfold_iplddt,Mean_Score
0,mcmahon,original,1,4.310000e-07,NaN,Nb.b201_HSA_404-609,0.828829,0.672600,0.3520,0.2994,...,0.717862,0.1053,0.005310,0.886037,95.894167,0.669827,0.8469,8540.50,NaN,1.460196
1,mcmahon,original,1,4.310000e-07,NaN,Nb.b201_HSA_404-601,0.531820,0.603329,0.1551,0.2819,...,0.713582,0.1133,0.005263,0.878948,95.717916,0.625092,0.8463,65086.93,NaN,0.554766
2,mcmahon,original,1,4.310000e-07,NaN,Nb.b201_HSA_25-609,0.644171,0.670695,0.2906,0.1625,...,0.538915,0.2691,0.005231,0.877172,86.205385,0.000000,0.4583,41598.65,63.759457,1.858561
3,mcmahon,original,1,2.300000e-06,NaN,Nb.AQ100_Adiponectin,0.163092,0.281817,0.0822,0.1487,...,0.370364,0.0486,0.028691,0.882637,84.943667,0.006573,0.3299,26952.43,56.575943,-0.364176
4,mcmahon,original,0,NaN,NaN,Nb.AQ101_Adiponectin,0.201847,0.338975,0.2525,0.0573,...,0.244249,0.0466,0.040458,0.881152,81.203705,0.005250,0.2045,15987.81,46.311729,-0.731116


In [6]:
# Product (Geometric approach)
df['Product_Score'] = df_norm.prod(axis=1)
df.head()

,source,type,binder,KD[M],EC50[M],sample,chai_constrained_ipsae,chai_constrained_interface_iptm,chai_constrained_pdockq,af3_pdockq,...,chai_unconstrained_pdockq,chai_unconstrained_ipsae,chai_unconstrained_per_chain_ptm,cf_iplddt,af3_ipsae,cf_pdockq2,cf_pyros_binder_score,esmfold_iplddt,Mean_Score,Product_Score
0,mcmahon,original,1,4.310000e-07,NaN,Nb.b201_HSA_404-609,0.828829,0.672600,0.3520,0.2994,...,0.1053,0.005310,0.886037,95.894167,0.669827,0.8469,8540.50,NaN,1.460196,1.700506
1,mcmahon,original,1,4.310000e-07,NaN,Nb.b201_HSA_404-601,0.531820,0.603329,0.1551,0.2819,...,0.1133,0.005263,0.878948,95.717916,0.625092,0.8463,65086.93,NaN,0.554766,0.273392
2,mcmahon,original,1,4.310000e-07,NaN,Nb.b201_HSA_25-609,0.644171,0.670695,0.2906,0.1625,...,0.2691,0.005231,0.877172,86.205385,0.000000,0.4583,41598.65,63.759457,1.858561,3.097247
3,mcmahon,original,1,2.300000e-06,NaN,Nb.AQ100_Adiponectin,0.163092,0.281817,0.0822,0.1487,...,0.0486,0.028691,0.882637,84.943667,0.006573,0.3299,26952.43,56.575943,-0.364176,-0.233669
4,mcmahon,original,0,NaN,NaN,Nb.AQ101_Adiponectin,0.201847,0.338975,0.2525,0.0573,...,0.0466,0.040458,0.881152,81.203705,0.005250,0.2045,15987.81,46.311729,-0.731116,0.531098


In [15]:
df.to_csv("score_combi_test.csv")

In [ ]:
# Sort by your new consensus
df = df.sort_values(by='Product_Score', ascending=False)